# Thực nghiệm ETS và Croston trên 10 chuỗi (C01)
**Học phần:** Phân tích dữ liệu kinh doanh — IS403.R11. **Nhóm thực hiện:** Nhóm 6.
Đo thời gian dự báo của ETS (`AutoETS`, chọn cấu hình theo AICc) và Croston cổ điển trên 10 chuỗi sản phẩm–cửa hàng. Cả hai dùng chu kỳ 7 ngày, lịch sử `d_1–d_1885` và chân trời dự báo 28 ngày; không dùng Validation hoặc Test.
Mỗi lần đo gồm một lần huấn luyện và dự báo cho một chuỗi, không gồm đọc dữ liệu hoặc ghi tệp. Kết quả dùng để chốt cỡ mẫu (C05), chưa đánh giá độ chính xác.

In [ ]:
import os, sys, subprocess, platform, json, zipfile, gc
from pathlib import Path
from datetime import datetime, timezone, timedelta

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    runtime, base = 'kaggle', Path('/kaggle/working')
elif 'COLAB_RELEASE_TAG' in os.environ:
    runtime, base = 'colab', Path('/content')
else:
    raise RuntimeError('Chạy notebook này trên Colab hoặc Kaggle; dùng lệnh README trên máy cá nhân.')
ROOT = base / 'IS403.R11-M5-Competition'
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/haiphamt/IS403.R11-M5-Competition.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'PyYAML', 'statsforecast'], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Runtime:', runtime)

## Dữ liệu nghiên cứu
Notebook đọc 10 chuỗi đầu tiên trong `sales_train_validation.csv` từ bộ M5 gốc. Trên Colab, tải lên một tệp ZIP chứa đủ 5 tệp CSV; trên Kaggle, gắn bộ dữ liệu cuộc thi vào Input.

In [ ]:
if runtime == 'colab':
    from google.colab import files
    names = ['calendar.csv', 'sell_prices.csv', 'sales_train_validation.csv', 'sales_train_evaluation.csv', 'sample_submission.csv']
    candidates = []
    for path in base.glob('*.zip'):
        if zipfile.is_zipfile(path):
            with zipfile.ZipFile(path) as archive:
                if set(names).issubset(archive.namelist()):
                    candidates.append(path)
    if not candidates:
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded]
        del uploaded
    if len(candidates) != 1:
        raise ValueError('Cần đúng một ZIP chứa đủ 5 file M5 gốc.')
    with zipfile.ZipFile(candidates[0]) as archive:
        if not set(names).issubset(archive.namelist()):
            raise ValueError('ZIP thiếu file M5 hoặc file CSV không nằm ngay ở thư mục gốc.')
        archive.extractall(ROOT / 'data' / 'raw', members=names)
    gc.collect()
else:
    print('Kaggle: đọc bộ M5 đã gắn trong Input.')

In [ ]:
import json
import pandas as pd

subprocess.run([sys.executable, 'scripts/benchmark_ets_croston.py'], check=True)

out = ROOT / 'results' / 'tables'
print(pd.read_csv(out / f'c01_ets_croston_runtime_{runtime}.csv').groupby('model')['seconds_per_series'].describe())
summary = json.loads((out / f'c01_ets_croston_summary_{runtime}.json').read_text(encoding='utf-8'))
print(json.dumps(summary['per_model'], indent=2))
subprocess.run(['free', '-h'], check=True)
subprocess.run(['nproc'], check=True)

RESULT_ZIP = base / f'is403_c01_ets_croston_{runtime}.zip'
with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(out.glob(f'c01_*_{runtime}.*')):
        archive.write(path, arcname=path.name)
print('Results:', RESULT_ZIP)

In [ ]:
if runtime == 'colab':
    files.download(str(RESULT_ZIP))
else:
    print('Tải ZIP kết quả từ Output của notebook:', RESULT_ZIP.name)